In [ ]:
import math

from mealpy.optimizer import Optimizer

In [ ]:
class OptimizerK(Optimizer):
    """
    Optimizer K
    """

    def __init__(self, epoch=2000, pop_size=100, h_memory=6, p_best=0.11,
                 elite_frac=0.1, restart_patience=200, ls_budget=60, **kwargs):
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 1000000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [10, 10000])
        self.h_memory = self.validator.check_int("h_memory", h_memory, [2, 50])
        self.p_best = self.validator.check_float("p_best", p_best, (0.0, 1.0))
        self.elite_frac = self.validator.check_float("elite_frac", elite_frac, (0.0, 0.5))
        self.restart_patience = self.validator.check_int("restart_patience", restart_patience, [10, 100000])
        self.ls_budget = self.validator.check_int("ls_budget", ls_budget, [0, 10000])
        self.set_parameters(["epoch", "pop_size", "h_memory", "p_best",
                             "elite_frac", "restart_patience", "ls_budget"])
        self.sort_flag = False

    def initialize_variables(self):
        self.lo = self.problem.lb
        self.hi = self.problem.ub
        self.ndim = self.problem.n_dims
        self.pop_min = max(8, self.pop_size // 5)
        self.mF = np.full(self.h_memory, 0.5)
        self.mCR = np.full(self.h_memory, 0.5)
        self.mem_idx = 0
        self.archive = np.empty((0, self.ndim))
        self.stag = 0
        self.last_best = np.inf

    def initialization(self):
        if self.pop is None:
            n = self.pop_size
            cut = np.linspace(0, 1, n + 1)
            rd = cut[:n, None] + np.random.rand(n, self.ndim) * (cut[1:, None] - cut[:n, None])

            for j in range(self.ndim):
                rd[:, j] = np.random.permutation(rd[:, j])
            positions = self.lo + rd * (self.hi - self.lo)

            self.pop = [self.generate_agent(positions[i]) for i in range(n)]

    def evolve(self, epoch):
        progress = (epoch - 1) / max(self.epoch - 1, 1)
        n_target = max(self.pop_min, int(round(self.pop_size - (self.pop_size - self.pop_min) * progress)))
        if len(self.pop) > n_target:
            self.pop = self.get_sorted_population(self.pop, self.problem.minmax)[:n_target]

        n = len(self.pop)
        pos = np.array([a.solution for a in self.pop])
        fit = np.array([a.target.fitness for a in self.pop])

        idx = np.random.randint(0, self.h_memory, n)
        F = self.mF[idx] + 0.1 * np.tan(np.pi * (np.random.rand(n) - 0.5))

        for _ in range(8):
            bad = F <= 0

            if not bad.any():
                break

            F[bad] = self.mF[idx[bad]] + 0.1 * np.tan(np.pi * (np.random.rand(bad.sum()) - 0.5))

        F = np.clip(F, 1e-6, 1.0)
        CR = np.clip(np.random.normal(self.mCR[idx], 0.1), 0, 1)

        order = np.argsort(fit)
        p_top = max(2, int(round(self.p_best * n)))
        pbest = order[:p_top][np.random.randint(0, p_top, n)]
        union = pos if self.archive.shape[0] == 0 else np.vstack([pos, self.archive])

        r1 = np.random.randint(0, n, n)
        r2 = np.random.randint(0, union.shape[0], n)

        for _ in range(5):
            bad = r1 == np.arange(n)

            if bad.any():
                r1[bad] = np.random.randint(0, n, bad.sum())

            bad = (r2 == np.arange(n)) | (r2 == r1)

            if not bad.any():
                break

            r2[bad] = np.random.randint(0, union.shape[0], bad.sum())

        mutant = pos + F[:, None] * (pos[pbest] - pos + pos[r1] - union[r2])
        mutant = np.where(mutant < self.lo, (pos + self.lo) / 2, mutant)
        mutant = np.where(mutant > self.hi, (pos + self.hi) / 2, mutant)
        mask = np.random.rand(n, self.ndim) < CR[:, None]
        mask[np.arange(n), np.random.randint(0, self.ndim, n)] = True
        trial = np.where(mask, mutant, pos)

        new_pop = [self.generate_agent(self.correct_solution(trial[i])) for i in range(n)]

        improved = np.zeros(n, dtype=bool)
        delta = np.zeros(n)

        for i in range(n):
            if self.compare_target(new_pop[i].target, self.pop[i].target, self.problem.minmax):
                improved[i] = True
                delta[i] = abs(self.pop[i].target.fitness - new_pop[i].target.fitness)

        if improved.any():
            self.archive = np.vstack([self.archive, pos[improved]]) if self.archive.size else pos[improved].copy()

            if self.archive.shape[0] > n:
                self.archive = self.archive[np.random.choice(self.archive.shape[0], n, replace=False)]

            if delta.sum() > 0:
                w = delta[improved] / delta[improved].sum()
                sF, sCR = F[improved], CR[improved]
                denom = np.sum(w * sF)

                if denom > 1e-30:
                    self.mF[self.mem_idx] = float(np.sum(w * sF * sF) / denom)
                    self.mCR[self.mem_idx] = float(np.sum(w * sCR * sCR) / max(np.sum(w * sCR), 1e-30))
                    self.mem_idx = (self.mem_idx + 1) % self.h_memory

        self.pop = [new_pop[i] if improved[i] else self.pop[i] for i in range(n)]

        best_fit = min(a.target.fitness for a in self.pop)

        if abs(best_fit - self.last_best) < 1e-12:
            self.stag += 1
        else:
            self.stag = 0

        self.last_best = best_fit

        if self.stag >= self.restart_patience:
            sorted_pop = self.get_sorted_population(self.pop, self.problem.minmax)
            keep = max(2, n // 10)
            elite = sorted_pop[:keep]
            fresh = [self.generate_agent() for _ in range(n - keep)]

            self.pop = elite + fresh
            self.archive = np.empty((0, self.ndim))
            self.stag = 0

    def track_optimize_process(self):
        if self.ls_budget > 0:
            best_x = self.g_best.solution.copy()
            step = 0.05 * (self.hi - self.lo)
            no_imp = 0

            for _ in range(self.ls_budget):
                imp = False

                for j in np.random.permutation(self.ndim):
                    for sign in (-1.0, 1.0):
                        t = best_x.copy()
                        t[j] = np.clip(best_x[j] + sign * step[j], self.lo[j], self.hi[j])
                        agent = self.generate_agent(self.correct_solution(t))

                        if self.compare_target(agent.target, self.g_best.target, self.problem.minmax):
                            best_x = t
                            self.g_best = agent
                            imp = True

                            break

                step *= 1.2 if imp else 0.5
                no_imp = 0 if imp else no_imp + 1

                if no_imp >= 3 and np.all(step < 1e-12 * (self.hi - self.lo + 1e-30)):
                    break

        super().track_optimize_process()

In [ ]:
class OptimizerKF(Optimizer):
    """
    Optimizer K
    """

    def __init__(self, epoch=2000, pop_size=100, pop_min=20,
                 de_iters=60, pso_epochs=20, h_memory=6, p_best=0.11,
                 elite_frac=0.1, restart_patience=200, ls_budget=60, **kwargs):
        super().__init__(**kwargs)

        self.epoch = self.validator.check_int("epoch", epoch, [1, 1000000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [10, 10000])
        self.pop_min = self.validator.check_int("pop_min", pop_min, [4, 10000])
        self.de_iters = self.validator.check_int("de_iters", de_iters, [1, 1000])
        self.pso_epochs = self.validator.check_int("pso_epochs", pso_epochs, [1, 1000])
        self.h_memory = self.validator.check_int("h_memory", h_memory, [2, 50])
        self.p_best = self.validator.check_float("p_best", p_best, (0.0, 1.0))
        self.elite_frac = self.validator.check_float("elite_frac", elite_frac, (0.0, 0.5))
        self.restart_patience = self.validator.check_int("restart_patience", restart_patience, [10, 100000])
        self.ls_budget = self.validator.check_int("ls_budget", ls_budget, [0, 10000])

        self.set_parameters(["epoch", "pop_size", "pop_min", "de_iters", "pso_epochs",
                             "h_memory", "p_best", "elite_frac", "restart_patience", "ls_budget"])
        self.sort_flag = False

    def initialize_variables(self):
        self.lo = self.problem.lb
        self.hi = self.problem.ub
        self.ndim = self.problem.n_dims
        self.diag = float(np.linalg.norm(self.hi - self.lo))
        self.mF = np.full(self.h_memory, 0.5)
        self.mCR = np.full(self.h_memory, 0.5)
        self.mem_idx = 0
        self.archive = np.empty((0, self.ndim))
        self.stag = 0
        self.last_best = np.inf
        self.rad_factor = 0.5

    def initialization(self):
        if self.pop is None:
            n = self.pop_size
            cut = np.linspace(0, 1, n + 1)
            rd = cut[:n, None] + np.random.rand(n, self.ndim) * (cut[1:, None] - cut[:n, None])

            for j in range(self.ndim):
                rd[:, j] = np.random.permutation(rd[:, j])

            positions = self.lo + rd * (self.hi - self.lo)

            self.pop = [self.generate_agent(positions[i]) for i in range(n)]

    def evolve(self, epoch):
        progress = (epoch - 1) / max(self.epoch - 1, 1)
        n_target = max(self.pop_min, int(round(self.pop_size - (self.pop_size - self.pop_min) * progress)))

        if len(self.pop) > n_target:
            self.pop = self.get_sorted_population(self.pop, self.problem.minmax)[:n_target]

            if self.archive.shape[0] > n_target:
                self.archive = self.archive[np.random.choice(self.archive.shape[0], n_target, replace=False)]

        for _ in range(self.de_iters):
            n = len(self.pop)
            pos = np.array([a.solution for a in self.pop])
            fit = np.array([a.target.fitness for a in self.pop])

            idx = np.random.randint(0, self.h_memory, n)
            F = self.mF[idx] + 0.1 * np.tan(np.pi * (np.random.rand(n) - 0.5))

            for _ in range(8):

                bad = F <= 0
                if not bad.any():
                    break

                F[bad] = self.mF[idx[bad]] + 0.1 * np.tan(np.pi * (np.random.rand(bad.sum()) - 0.5))

            F = np.clip(F, 1e-6, 1.0)
            CR = np.clip(np.random.normal(self.mCR[idx], 0.1), 0, 1)

            order = np.argsort(fit)
            p_top = max(2, int(round(self.p_best * n)))
            pbest = order[:p_top][np.random.randint(0, p_top, n)]
            union = pos if self.archive.shape[0] == 0 else np.vstack([pos, self.archive])

            r1 = np.random.randint(0, n, n)
            r2 = np.random.randint(0, union.shape[0], n)

            for _ in range(5):
                bad = r1 == np.arange(n)

                if bad.any():
                    r1[bad] = np.random.randint(0, n, bad.sum())

                bad = (r2 == np.arange(n)) | (r2 == r1)

                if not bad.any():
                    break

                r2[bad] = np.random.randint(0, union.shape[0], bad.sum())

            mutant = pos + F[:, None] * (pos[pbest] - pos + pos[r1] - union[r2])
            mutant = np.where(mutant < self.lo, (pos + self.lo) / 2, mutant)
            mutant = np.where(mutant > self.hi, (pos + self.hi) / 2, mutant)

            mask = np.random.rand(n, self.ndim) < CR[:, None]
            mask[np.arange(n), np.random.randint(0, self.ndim, n)] = True
            trial = np.where(mask, mutant, pos)

            new_agents = [self.generate_agent(self.correct_solution(trial[i])) for i in range(n)]

            improved = np.zeros(n, dtype=bool)
            delta = np.zeros(n)

            for i in range(n):
                if self.compare_target(new_agents[i].target, self.pop[i].target, self.problem.minmax):
                    improved[i] = True
                    delta[i] = abs(self.pop[i].target.fitness - new_agents[i].target.fitness)

            if improved.any():
                self.archive = np.vstack([self.archive, pos[improved]]) if self.archive.size else pos[improved].copy()

                if self.archive.shape[0] > n:
                    self.archive = self.archive[np.random.choice(self.archive.shape[0], n, replace=False)]

                if delta.sum() > 0:
                    w = delta[improved] / delta[improved].sum()
                    sF, sCR = F[improved], CR[improved]
                    denom_f = np.sum(w * sF)

                    if denom_f > 1e-30:
                        self.mF[self.mem_idx] = float(np.sum(w * sF * sF) / denom_f)
                        self.mCR[self.mem_idx] = float(np.sum(w * sCR * sCR) / max(np.sum(w * sCR), 1e-30))
                        self.mem_idx = (self.mem_idx + 1) % self.h_memory

            self.pop = [new_agents[i] if improved[i] else self.pop[i] for i in range(n)]

        n = len(self.pop)
        elite_n = max(2, int(self.elite_frac * n))
        sorted_pop = self.get_sorted_population(self.pop, self.problem.minmax)
        elite = sorted_pop[:elite_n]

        radius = max(self.rad_factor * self.diag * (1 - 0.7 * progress), 1e-6 * self.diag)
        thickness = radius * 0.5
        centroid = np.array(sorted_pop[0].solution)

        n_explore = n - elite_n
        directions = np.random.normal(0, 1, (n_explore, self.ndim))
        directions /= np.maximum(np.linalg.norm(directions, axis=1, keepdims=True), 1e-12)

        r_min = max(radius - thickness, 0.0)

        u = np.random.rand(n_explore, 1)

        if radius > 0:
            ratio_d = (r_min / radius) ** self.ndim
            r = radius * (u * (1.0 - ratio_d) + ratio_d) ** (1.0 / self.ndim)
        else:
            r = np.zeros((n_explore, 1))

        explorers = directions * r + centroid
        explorers = np.clip(explorers, self.lo, self.hi)

        pso_pop = [a for a in elite] + [self.generate_agent(self.correct_solution(explorers[i])) for i in
                                        range(n_explore)]
        pso_pos = np.array([a.solution for a in pso_pop])
        pso_fit = np.array([a.target.fitness for a in pso_pop])

        v_max = 0.5 * (self.hi - self.lo)
        speed = np.random.uniform(-1, 1, (n, self.ndim)) * v_max
        pbest_pos = pso_pos.copy()
        pbest_fit = pso_fit.copy()
        pbest_agents = list(pso_pop)

        gbest_agent = self.get_best_agent(pso_pop, self.problem.minmax)
        gbest_pos = np.array(gbest_agent.solution)
        gbest_fit = gbest_agent.target.fitness

        pso_stag = 0
        prev_gbest = gbest_fit

        for t in range(self.pso_epochs):
            r = t / max(self.pso_epochs - 1, 1)
            c1 = 2.5 - 2.0 * r
            c2 = 0.5 + 2.0 * r

            speed = (c1 * np.random.rand(n, self.ndim) * (pbest_pos - pso_pos)
                     + c2 * np.random.rand(n, self.ndim) * (gbest_pos - pso_pos))
            speed = np.clip(speed, -v_max, v_max)

            new_pos = pso_pos + speed
            out = (new_pos < self.lo) | (new_pos > self.hi)
            new_pos = np.clip(new_pos, self.lo, self.hi)
            speed = np.where(out, -0.5 * speed, speed)

            new_agents = [self.generate_agent(self.correct_solution(new_pos[i])) for i in range(n)]
            pso_pos = new_pos
            pso_fit = np.array([a.target.fitness for a in new_agents])

            for i in range(n):
                if self.compare_target(new_agents[i].target, pbest_agents[i].target, self.problem.minmax):
                    pbest_agents[i] = new_agents[i]
                    pbest_pos[i] = new_pos[i]
                    pbest_fit[i] = pso_fit[i]

                    if self.compare_target(new_agents[i].target, gbest_agent.target, self.problem.minmax):
                        gbest_agent = new_agents[i]
                        gbest_pos = new_pos[i].copy()
                        gbest_fit = pso_fit[i]

            if abs(prev_gbest - gbest_fit) < 1e-12:
                pso_stag += 1
            else:
                pso_stag = 0

            prev_gbest = gbest_fit

            if pso_stag >= max(3, self.pso_epochs // 5):
                worst_n = max(1, n // 4)
                worst_idx = np.argsort(pbest_fit)[-worst_n:]
                beta = 1.5
                sigma_u = (math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
                           / (math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))) ** (1 / beta)
                u_lev = np.random.normal(0, sigma_u, (worst_n, self.ndim))
                v_lev = np.random.normal(0, 1, (worst_n, self.ndim))
                step = 0.1 * (self.hi - self.lo) * u_lev / (np.abs(v_lev) ** (1 / beta) + 1e-12)
                new_jumped = np.clip(gbest_pos + step, self.lo, self.hi)

                for k, idx_w in enumerate(worst_idx):
                    pso_pos[idx_w] = new_jumped[k]
                    speed[idx_w] = np.random.uniform(-1, 1, self.ndim) * v_max

                pso_stag = 0

        de_order = np.argsort([a.target.fitness for a in self.pop])
        pso_order = np.argsort(pbest_fit)
        keep_de = max(elite_n, n // 2)

        mixed = ([self.pop[i] for i in de_order[:keep_de]]
                 + [pbest_agents[i] for i in pso_order[:n - keep_de]])

        self.pop = mixed

        best_now = self.get_best_agent(self.pop, self.problem.minmax).target.fitness

        if best_now < self.last_best - 1e-12:
            self.stag = 0
            self.rad_factor = min(0.7, self.rad_factor * 1.05)
        else:
            self.stag += 1
            self.rad_factor *= 0.85

        self.last_best = best_now

        if self.stag >= self.restart_patience:
            sorted_pop = self.get_sorted_population(self.pop, self.problem.minmax)
            keep = max(2, n // 10)
            elite = sorted_pop[:keep]
            fresh = [self.generate_agent() for _ in range(n - keep)]

            self.pop = elite + fresh
            self.archive = np.empty((0, self.ndim))
            self.stag = 0
            self.rad_factor = 0.5

        if epoch == self.epoch and self.ls_budget > 0:
            best_x = np.array(self.g_best.solution, dtype=float).copy()
            best_agent = self.g_best
            step = 0.05 * (self.hi - self.lo)
            no_imp = 0

            for _ in range(self.ls_budget):
                imp = False
                for j in np.random.permutation(self.ndim):
                    for sign in (-1.0, 1.0):
                        trial = best_x.copy()
                        trial[j] = np.clip(best_x[j] + sign * step[j], self.lo[j], self.hi[j])

                        cand = self.generate_agent(self.correct_solution(trial))

                        if self.compare_target(cand.target, best_agent.target, self.problem.minmax):
                            best_x = trial
                            best_agent = cand
                            worst_idx = int(np.argmax([a.target.fitness for a in self.pop]))

                            self.pop[worst_idx] = cand

                            imp = True
                            break
                    if imp:
                        break

                step *= 1.2 if imp else 0.5
                no_imp = 0 if imp else no_imp + 1

                if no_imp >= 3 and np.all(step < 1e-12 * (self.hi - self.lo + 1e-30)):
                    break

In [ ]:
class OptimizerKHG(Optimizer):
    """
    Optimizer K
    """

    def __init__(self, epoch=60, pop_size=100, pop_min=20,
                 de_iters=60, pso_epochs=20, h_memory=6, p_best=0.11,
                 elite_frac=0.1, restart_patience=12, ls_budget=60, **kwargs):
        super().__init__(**kwargs)

        self.epoch = self.validator.check_int("epoch", epoch, [1, 1000000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [10, 10000])
        self.pop_min = self.validator.check_int("pop_min", pop_min, [4, 10000])
        self.de_iters = self.validator.check_int("de_iters", de_iters, [1, 1000])
        self.pso_epochs = self.validator.check_int("pso_epochs", pso_epochs, [1, 1000])
        self.h_memory = self.validator.check_int("h_memory", h_memory, [2, 50])
        self.p_best = self.validator.check_float("p_best", p_best, (0.0, 1.0))
        self.elite_frac = self.validator.check_float("elite_frac", elite_frac, (0.0, 0.5))
        self.restart_patience = self.validator.check_int("restart_patience", restart_patience, [3, 100000])
        self.ls_budget = self.validator.check_int("ls_budget", ls_budget, [0, 10000])

        self.set_parameters(["epoch", "pop_size", "pop_min", "de_iters", "pso_epochs",
                             "h_memory", "p_best", "elite_frac", "restart_patience", "ls_budget"])
        self.sort_flag = False

    def _hypersphere_sample(self, n, radius, thickness, centroid):
        directions = np.random.normal(0, 1, (n, self.ndim))
        directions /= np.maximum(np.linalg.norm(directions, axis=1, keepdims=True), 1e-12)

        r_min = max(radius - thickness, 0.0)

        u = np.random.rand(n, 1)
        r = (u * (radius ** self.ndim - r_min ** self.ndim) + r_min ** self.ndim) ** (1.0 / self.ndim)

        return directions * r + centroid

    def _auto_radius(self, positions, centroid, progress=0.0):
        if positions.shape[0] < 2:
            return 0.5 * self.diag, 0.16 * self.diag

        sigma = float(np.mean(np.std(positions, axis=0)))

        d_from_center = np.linalg.norm(positions - centroid, axis=1)
        d_avg = float(np.mean(d_from_center))

        radius = max(sigma * math.sqrt(self.ndim), d_avg)
        floor = (0.5 - 0.49 * progress) * self.diag

        radius = max(radius, floor)
        radius = min(radius, self.diag)

        thickness = radius / 3.0

        return radius, thickness

    def initialize_variables(self):
        self.lo = self.problem.lb
        self.hi = self.problem.ub
        self.ndim = self.problem.n_dims
        self.diag = float(np.linalg.norm(self.hi - self.lo))
        self.mF = np.full(self.h_memory, 0.5)
        self.mCR = np.full(self.h_memory, 0.5)
        self.mem_idx = 0
        self.archive = np.empty((0, self.ndim))
        self.stag = 0
        self.last_best = np.inf

    def initialization(self):
        if self.pop is None:
            n = self.pop_size
            domain_center = 0.5 * (self.lo + self.hi)

            init_radius = 0.5 * self.diag
            init_thickness = init_radius / 3.0

            positions = self._hypersphere_sample(n, init_radius, init_thickness, domain_center)
            positions = np.clip(positions, self.lo, self.hi)

            self.pop = [self.generate_agent(positions[i]) for i in range(n)]

    def evolve(self, epoch):
        progress = (epoch - 1) / max(self.epoch - 1, 1)
        n_target = max(self.pop_min, int(round(self.pop_size - (self.pop_size - self.pop_min) * progress)))

        if len(self.pop) > n_target:
            self.pop = self.get_sorted_population(self.pop, self.problem.minmax)[:n_target]

            if self.archive.shape[0] > n_target:
                self.archive = self.archive[np.random.choice(self.archive.shape[0], n_target, replace=False)]

        for _ in range(self.de_iters):
            n = len(self.pop)
            pos = np.array([a.solution for a in self.pop])
            fit = np.array([a.target.fitness for a in self.pop])

            idx = np.random.randint(0, self.h_memory, n)
            F = self.mF[idx] + 0.1 * np.tan(np.pi * (np.random.rand(n) - 0.5))

            for _ in range(8):
                bad = F <= 0

                if not bad.any():
                    break

                F[bad] = self.mF[idx[bad]] + 0.1 * np.tan(np.pi * (np.random.rand(bad.sum()) - 0.5))

            F = np.clip(F, 1e-6, 1.0)
            CR = np.clip(np.random.normal(self.mCR[idx], 0.1), 0, 1)

            order = np.argsort(fit)
            p_top = max(2, int(round(self.p_best * n)))
            pbest = order[:p_top][np.random.randint(0, p_top, n)]
            union = pos if self.archive.shape[0] == 0 else np.vstack([pos, self.archive])

            r1 = np.random.randint(0, n, n)
            r2 = np.random.randint(0, union.shape[0], n)

            for _ in range(5):
                bad = r1 == np.arange(n)

                if bad.any():
                    r1[bad] = np.random.randint(0, n, bad.sum())

                bad = (r2 == np.arange(n)) | (r2 == r1)
                if not bad.any():
                    break

                r2[bad] = np.random.randint(0, union.shape[0], bad.sum())

            mutant = pos + F[:, None] * (pos[pbest] - pos + pos[r1] - union[r2])
            mutant = np.where(mutant < self.lo, (pos + self.lo) / 2, mutant)
            mutant = np.where(mutant > self.hi, (pos + self.hi) / 2, mutant)

            mask = np.random.rand(n, self.ndim) < CR[:, None]
            mask[np.arange(n), np.random.randint(0, self.ndim, n)] = True
            trial = np.where(mask, mutant, pos)

            new_agents = [self.generate_agent(self.correct_solution(trial[i])) for i in range(n)]

            improved = np.zeros(n, dtype=bool)
            delta = np.zeros(n)

            for i in range(n):
                if self.compare_target(new_agents[i].target, self.pop[i].target, self.problem.minmax):
                    improved[i] = True
                    delta[i] = abs(self.pop[i].target.fitness - new_agents[i].target.fitness)

            if improved.any():
                self.archive = np.vstack([self.archive, pos[improved]]) if self.archive.size else pos[improved].copy()
                if self.archive.shape[0] > n:
                    self.archive = self.archive[np.random.choice(self.archive.shape[0], n, replace=False)]

                if delta.sum() > 0:
                    w = delta[improved] / delta[improved].sum()
                    sF, sCR = F[improved], CR[improved]
                    denom_f = np.sum(w * sF)

                    if denom_f > 1e-30:
                        self.mF[self.mem_idx] = float(np.sum(w * sF * sF) / denom_f)
                        self.mCR[self.mem_idx] = float(np.sum(w * sCR * sCR) / max(np.sum(w * sCR), 1e-30))
                        self.mem_idx = (self.mem_idx + 1) % self.h_memory

            self.pop = [new_agents[i] if improved[i] else self.pop[i] for i in range(n)]

        n = len(self.pop)
        elite_n = max(2, int(self.elite_frac * n))
        sorted_pop = self.get_sorted_population(self.pop, self.problem.minmax)
        elite = sorted_pop[:elite_n]
        centroid = np.array(sorted_pop[0].solution)

        current_positions = np.array([a.solution for a in self.pop])
        radius, thickness = self._auto_radius(current_positions, centroid, progress)

        n_explore = n - elite_n
        explorers = self._hypersphere_sample(n_explore, radius, thickness, centroid)
        explorers = np.clip(explorers, self.lo, self.hi)

        pso_pop = list(elite) + [self.generate_agent(self.correct_solution(explorers[i])) for i in range(n_explore)]
        pso_pos = np.array([a.solution for a in pso_pop])
        pso_fit = np.array([a.target.fitness for a in pso_pop])

        v_max = 0.5 * (self.hi - self.lo)
        speed = np.random.uniform(-1, 1, (n, self.ndim)) * v_max
        pbest_pos = pso_pos.copy()
        pbest_fit = pso_fit.copy()
        pbest_agents = list(pso_pop)

        gbest_agent = self.get_best_agent(pso_pop, self.problem.minmax)
        gbest_pos = np.array(gbest_agent.solution)
        gbest_fit = gbest_agent.target.fitness

        pso_stag = 0
        prev_gbest = gbest_fit

        for t in range(self.pso_epochs):
            r = t / max(self.pso_epochs - 1, 1)
            c1 = 2.5 - 2.0 * r
            c2 = 0.5 + 2.0 * r

            speed = (c1 * np.random.rand(n, self.ndim) * (pbest_pos - pso_pos)
                     + c2 * np.random.rand(n, self.ndim) * (gbest_pos - pso_pos))
            speed = np.clip(speed, -v_max, v_max)

            new_pos = pso_pos + speed
            out = (new_pos < self.lo) | (new_pos > self.hi)
            new_pos = np.clip(new_pos, self.lo, self.hi)
            speed = np.where(out, -0.5 * speed, speed)

            new_agents = [self.generate_agent(self.correct_solution(new_pos[i])) for i in range(n)]
            pso_pos = new_pos
            pso_fit = np.array([a.target.fitness for a in new_agents])

            for i in range(n):
                if self.compare_target(new_agents[i].target, pbest_agents[i].target, self.problem.minmax):
                    pbest_agents[i] = new_agents[i]
                    pbest_pos[i] = new_pos[i]
                    pbest_fit[i] = pso_fit[i]
                    if self.compare_target(new_agents[i].target, gbest_agent.target, self.problem.minmax):
                        gbest_agent = new_agents[i]
                        gbest_pos = new_pos[i].copy()
                        gbest_fit = pso_fit[i]

            if abs(prev_gbest - gbest_fit) < 1e-12:
                pso_stag += 1
            else:
                pso_stag = 0

            prev_gbest = gbest_fit

            if pso_stag >= max(3, self.pso_epochs // 5):
                worst_n = max(1, n // 4)
                worst_idx = np.argsort(pbest_fit)[-worst_n:]

                beta = 1.5
                sigma_u = (math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
                           / (math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))) ** (1 / beta)

                u_lev = np.random.normal(0, sigma_u, (worst_n, self.ndim))
                v_lev = np.random.normal(0, 1, (worst_n, self.ndim))

                step = 0.1 * (self.hi - self.lo) * u_lev / (np.abs(v_lev) ** (1 / beta) + 1e-12)

                new_jumped = np.clip(gbest_pos + step, self.lo, self.hi)

                for k, idx_w in enumerate(worst_idx):
                    pso_pos[idx_w] = new_jumped[k]
                    speed[idx_w] = np.random.uniform(-1, 1, self.ndim) * v_max

                pso_stag = 0

        de_order = np.argsort([a.target.fitness for a in self.pop])
        pso_order = np.argsort(pbest_fit)
        keep_de = max(elite_n, n // 2)

        self.pop = ([self.pop[i] for i in de_order[:keep_de]]
                    + [pbest_agents[i] for i in pso_order[:n - keep_de]])

        best_now = self.get_best_agent(self.pop, self.problem.minmax).target.fitness

        if best_now < self.last_best - 1e-12:
            self.stag = 0
        else:
            self.stag += 1

        self.last_best = best_now

        if self.stag >= self.restart_patience:
            sorted_pop = self.get_sorted_population(self.pop, self.problem.minmax)
            keep = max(2, n // 10)

            elite_seeds = sorted_pop[:keep]
            best_pos = np.array(elite_seeds[0].solution)

            current_positions = np.array([a.solution for a in self.pop])
            radius, thickness = self._auto_radius(current_positions, best_pos, progress=0.0)
            thickness = radius / 3.0

            fresh_pos = self._hypersphere_sample(n - keep, radius, thickness, best_pos)
            fresh_pos = np.clip(fresh_pos, self.lo, self.hi)

            fresh = [self.generate_agent(self.correct_solution(fresh_pos[i])) for i in range(n - keep)]

            self.pop = list(elite_seeds) + fresh
            self.archive = np.empty((0, self.ndim))
            self.stag = 0

        if epoch == self.epoch and self.ls_budget > 0:
            best_x = np.array(self.g_best.solution, dtype=float).copy()
            best_agent = self.g_best
            step = 0.05 * (self.hi - self.lo)
            no_imp = 0

            for _ in range(self.ls_budget):
                imp = False

                for j in np.random.permutation(self.ndim):
                    for sign in (-1.0, 1.0):
                        trial = best_x.copy()
                        trial[j] = np.clip(best_x[j] + sign * step[j], self.lo[j], self.hi[j])

                        cand = self.generate_agent(self.correct_solution(trial))

                        if self.compare_target(cand.target, best_agent.target, self.problem.minmax):
                            best_x = trial
                            best_agent = cand
                            worst_idx = int(np.argmax([a.target.fitness for a in self.pop]))

                            self.pop[worst_idx] = cand
                            imp = True

                            break
                    if imp:
                        break

                step *= 1.2 if imp else 0.5
                no_imp = 0 if imp else no_imp + 1

                if no_imp >= 3 and np.all(step < 1e-12 * (self.hi - self.lo + 1e-30)):
                    break

In [ ]:
import time
import numpy as np
import pandas as pd

import opfunu

from mealpy import FloatVar
from mealpy.swarm_based.PSO import OriginalPSO


def make_problem_dict(opfunu_func):
    return {
        "bounds": FloatVar(lb=opfunu_func.lb.tolist(), ub=opfunu_func.ub.tolist(), name="x"),
        "obj_func": opfunu_func.evaluate,
        "minmax": "min",
        "log_to": None,
    }


def run_trial(model_cls, model_kwargs, problem_dict, seed):
    model = model_cls(**model_kwargs)
    t0 = time.perf_counter()
    g_best = model.solve(problem_dict, seed=seed)
    elapsed = time.perf_counter() - t0
    return model, g_best, elapsed


def benchmark(algorithms, problems, n_runs=10, epoch=500, pop_size=40):
    summary_rows = []
    convergence_rows = []
    metrics_rows = []

    for prob_cls, ndim in problems:
        opf = prob_cls(ndim=ndim)
        problem_dict = make_problem_dict(opf)
        f_global = opf.f_global

        for algo_name, model_cls, extra_kwargs in algorithms:
            model_kwargs = {"epoch": epoch, "pop_size": pop_size, **extra_kwargs}

            best_fitnesses = []
            errors = []
            times = []
            convergences = []
            diversities = []
            explorations = []
            exploitations = []

            for seed in range(n_runs):
                model, g_best, elapsed = run_trial(model_cls, model_kwargs, problem_dict, seed)
                best_fit = float(g_best.target.fitness)
                best_fitnesses.append(best_fit)
                errors.append(abs(best_fit - f_global))
                times.append(elapsed)
                convergences.append(np.array(model.history.list_global_best_fit))
                diversities.append(np.array(model.history.list_diversity))
                explorations.append(np.array(model.history.list_exploration))
                exploitations.append(np.array(model.history.list_exploitation))

            best_fitnesses = np.array(best_fitnesses)
            errors = np.array(errors)
            times = np.array(times)

            summary_rows.append({
                "function": opf.name,
                "ndim": ndim,
                "algorithm": algo_name,
                "f_global": f_global,
                "best": best_fitnesses.min(),
                "mean": best_fitnesses.mean(),
                "median": np.median(best_fitnesses),
                "std": best_fitnesses.std(),
                "worst": best_fitnesses.max(),
                "mean_error": errors.mean(),
                "mean_time_s": times.mean(),
            })

            mean_curve = np.mean(np.array(convergences), axis=0)
            for i, val in enumerate(mean_curve):
                convergence_rows.append({
                    "function": opf.name,
                    "algorithm": algo_name,
                    "epoch": i + 1,
                    "mean_global_best": val,
                })

            metrics_rows.append({
                "function": opf.name,
                "algorithm": algo_name,
                "mean_diversity": np.mean(np.array(diversities)),
                "mean_exploration_pct": np.mean(np.array(explorations)),
                "mean_exploitation_pct": np.mean(np.array(exploitations)),
                "final_diversity": np.mean([d[-1] for d in diversities]),
            })

    return (
        pd.DataFrame(summary_rows),
        pd.DataFrame(convergence_rows),
        pd.DataFrame(metrics_rows),
    )


In [ ]:
NDIM = 20

problems = [
    (opfunu.name_based.Ackley01, NDIM),
    (opfunu.name_based.Alpine01, NDIM),
    (opfunu.name_based.ChungReynolds, NDIM),
    (opfunu.name_based.Griewank, NDIM),
    (opfunu.name_based.Salomon, NDIM),
    (opfunu.name_based.Zacharov, NDIM),
    (opfunu.cec_based.F92005, NDIM),
]

algorithms = [
    ("PSO", OriginalPSO, {}),
    ("K", OptimizerK, {"patience": 80}),
    ("KHG", OptimizerKHG, {})
]

pd.set_option("display.float_format", lambda v: f"{v:.4e}")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 20)

df_summary, df_conv, df_metrics = benchmark(
    algorithms, problems, n_runs=5, epoch=200, pop_size=40
)

print("=== SUMMARY ===")
print(df_summary.to_string(index=False))
print()

print("=== EXPLORATION / EXPLOITATION METRICS (mealpy history) ===")
print(df_metrics.to_string(index=False))
print()

print("=== HEAD-TO-HEAD WINNER (lower mean error) ===")
pivot = df_summary.pivot(index="function", columns="algorithm", values="mean_error")
pivot["winner"] = pivot.idxmin(axis=1)

pivot["PSO_over_K"] = pivot["PSO"] / pivot["K"]
pivot["PSO_over_KHG"] = pivot["PSO"] / pivot["KHG"]

pivot["K_over_PSO"] = pivot["K"] / pivot["PSO"]
pivot["K_over_KHG"] = pivot["K"] / pivot["KHG"]

pivot["KHG_over_K"] = pivot["KHG"] / pivot["K"]
pivot["KHG_over_PSO"] = pivot["KHG"] / pivot["PSO"]

print(pivot.to_string())
print()

In [ ]:
NRUNS = 5
NDIM = 30

problems = [
    (opfunu.name_based.Ackley01, NDIM),
    (opfunu.name_based.Alpine01, NDIM),
    (opfunu.name_based.ChungReynolds, NDIM),
    (opfunu.name_based.Griewank, NDIM),
    (opfunu.name_based.Salomon, NDIM),
    (opfunu.name_based.Zacharov, NDIM),
    (opfunu.cec_based.F92005, NDIM),
]

algorithms = [
    ("PSO", OriginalPSO, {}),
    ("K", OptimizerK, {"patience": 80}),
    ("KF", OptimizerKF, {}),
    ("KHG", OptimizerKHG, {})
]

pd.set_option("display.float_format", lambda v: f"{v:.4e}")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 20)

df_summary, df_conv, df_metrics = benchmark(
    algorithms, problems, n_runs=NRUNS, epoch=200, pop_size=40
)

print("=== SUMMARY ===")
print(df_summary.to_string(index=False))
print()

print("=== EXPLORATION / EXPLOITATION METRICS (mealpy history) ===")
print(df_metrics.to_string(index=False))
print()

print("=== HEAD-TO-HEAD WINNER (lower mean error) ===")
pivot = df_summary.pivot(index="function", columns="algorithm", values="mean_error")
pivot["winner"] = pivot.idxmin(axis=1)

pivot["PSO_over_K"] = pivot["PSO"] / pivot["K"]
pivot["PSO_over_KHG"] = pivot["PSO"] / pivot["KHG"]

pivot["K_over_PSO"] = pivot["K"] / pivot["PSO"]
pivot["K_over_KHG"] = pivot["K"] / pivot["KHG"]

pivot["KHG_over_K"] = pivot["KHG"] / pivot["K"]
pivot["KHG_over_PSO"] = pivot["KHG"] / pivot["PSO"]

print(pivot.to_string())
print()

In [10]:
NRUNS = 5
NDIM = 50

problems = [
    (opfunu.name_based.Ackley01, NDIM),
    (opfunu.name_based.Alpine01, NDIM),
    (opfunu.name_based.ChungReynolds, NDIM),
    (opfunu.name_based.Griewank, NDIM),
    (opfunu.name_based.Salomon, NDIM),
    (opfunu.name_based.Zacharov, NDIM),
    (opfunu.cec_based.F92005, NDIM),
]

algorithms = [
    ("PSO", OriginalPSO, {}),
    ("K", OptimizerK, {"patience": 80}),
    ("KHG", OptimizerKF, {})
]

pd.set_option("display.float_format", lambda v: f"{v:.4e}")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 20)

df_summary, df_conv, df_metrics = benchmark(
    algorithms, problems, n_runs=NRUNS, epoch=300, pop_size=40
)

print("=== SUMMARY ===")
print(df_summary.to_string(index=False))
print()

print("=== EXPLORATION / EXPLOITATION METRICS (mealpy history) ===")
print(df_metrics.to_string(index=False))
print()

print("=== HEAD-TO-HEAD WINNER (lower mean error) ===")
pivot = df_summary.pivot(index="function", columns="algorithm", values="mean_error")
pivot["winner"] = pivot.idxmin(axis=1)

pivot["PSO_over_K"] = pivot["PSO"] / pivot["K"]
pivot["PSO_over_KHG"] = pivot["PSO"] / pivot["KHG"]

pivot["K_over_PSO"] = pivot["K"] / pivot["PSO"]
pivot["K_over_KHG"] = pivot["K"] / pivot["KHG"]

pivot["KHG_over_K"] = pivot["KHG"] / pivot["K"]
pivot["KHG_over_PSO"] = pivot["KHG"] / pivot["PSO"]

print(pivot.to_string())
print()

__main__.OptimizerKF:  42%|████▏     | 125/300 [00:15<00:21,  8.27epoch/s, c_best=0.000000, g_best=0.000000]


KeyboardInterrupt: 

In [ ]:
NRUNS = 5
NDIM = 120

problems = [
    (opfunu.name_based.Ackley01, NDIM),
    (opfunu.name_based.Alpine01, NDIM),
    (opfunu.name_based.ChungReynolds, NDIM),
    (opfunu.name_based.Griewank, NDIM),
    (opfunu.name_based.Salomon, NDIM),
    (opfunu.name_based.Zacharov, NDIM),
]

algorithms = [
    ("PSO", OriginalPSO, {}),
    ("K", OptimizerK, {"patience": 80}),
    ("KHG", OptimizerKHG, {})
]

pd.set_option("display.float_format", lambda v: f"{v:.4e}")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 20)

df_summary, df_conv, df_metrics = benchmark(
    algorithms, problems, n_runs=NRUNS, epoch=250, pop_size=40
)

print("=== SUMMARY ===")
print(df_summary.to_string(index=False))
print()

print("=== EXPLORATION / EXPLOITATION METRICS (mealpy history) ===")
print(df_metrics.to_string(index=False))
print()

print("=== HEAD-TO-HEAD WINNER (lower mean error) ===")
pivot = df_summary.pivot(index="function", columns="algorithm", values="mean_error")
pivot["winner"] = pivot.idxmin(axis=1)

pivot["PSO_over_K"] = pivot["PSO"] / pivot["K"]
pivot["PSO_over_KHG"] = pivot["PSO"] / pivot["KHG"]

pivot["K_over_PSO"] = pivot["K"] / pivot["PSO"]
pivot["K_over_KHG"] = pivot["K"] / pivot["KHG"]

pivot["KHG_over_K"] = pivot["KHG"] / pivot["K"]
pivot["KHG_over_PSO"] = pivot["KHG"] / pivot["PSO"]

print(pivot.to_string())
print()